# Copy Delta table from one Databricks workspace to another

Copy a Delta table across two **separate** Databricks workspaces. Tables don't move between
workspaces directly — you move them through **shared cloud storage (ADLS Gen2)**.

**This copy:**

| | Workspace | Table |
|---|---|---|
| Source | `dbw-arc-merone-pipeline-prod` | `hive_metastore.input_database_incremental.fat_roads` |
| Target | `dbw-arc-merone-pipeline-dev` | `hive_metastore.apa_prod.fat_roads` |

Both workspaces are TomTom `arc-merone-pipeline`, so they can almost certainly reach a common
ADLS Gen2 account. Three approaches below, **best first**.

## Check these two things first

1. **ADLS access** — does the dev cluster's managed identity / service principal have
   `Storage Blob Data Reader` (or `Contributor` for writes) on the container you stage to?
   Cross-workspace copies fail here most often.
2. **The `id` struct column** — `DEEP CLONE` and Delta writes preserve nested structs
   automatically; no schema flattening needed. **Avoid** CSV / `CREATE TABLE AS SELECT` into
   non-Delta formats, which would break the struct.

> Fill in `<shared-container>` and `<storageaccount>` below from the `DESCRIBE DETAIL` output
> in Step 1 before running the write/clone cells.

---
## Option A — Shared ADLS path (recommended, works with hive_metastore)

Most reliable for `hive_metastore` tables. Stage the data to a container both workspaces own,
then deep-clone it into the target.

### Step 1 — On the **PROD** workspace: find the physical location

Note the `location` (an `abfss://...` path) and the `format` (should be `delta`).

In [ ]:
%sql
DESCRIBE DETAIL input_database_incremental.fat_roads;

### Step 2 — On the **PROD** workspace: write a copy to shared storage

Use a container/path that the **dev** cluster's identity is also granted on.

In [ ]:
# Run on the PROD workspace
src = spark.table("input_database_incremental.fat_roads")

staging_path = "abfss://<shared-container>@<storageaccount>.dfs.core.windows.net/exchange/fat_roads"

(src.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(staging_path))

print(f"Staged fat_roads to {staging_path}")

### Step 3 — On the **DEV** workspace: create the table in `apa_prod`

**`DEEP CLONE` is preferable** — it physically copies data, preserves the full Delta schema
(including the `id` struct), and is transactionally consistent.

In [ ]:
%sql
-- DEEP CLONE: managed/independent Delta copy (best)
CREATE TABLE hive_metastore.apa_prod.fat_roads
DEEP CLONE delta.`abfss://<shared-container>@<storageaccount>.dfs.core.windows.net/exchange/fat_roads`;

In [ ]:
%sql
-- Alternative: read the staged files in place (no physical copy into dev)
CREATE TABLE hive_metastore.apa_prod.fat_roads
USING DELTA
LOCATION 'abfss://<shared-container>@<storageaccount>.dfs.core.windows.net/exchange/fat_roads';

---
## Option B — Dev cluster can already read the prod table's storage path

If the `location` from Step 1's `DESCRIBE DETAIL` is a path the **dev** workspace also has ADLS
access to, skip staging entirely. This reads the prod table's underlying files directly and
copies them into dev — **no write needed on prod**.

Run on the **DEV** workspace (replace the location with the value from `DESCRIBE DETAIL`):

In [ ]:
%sql
CREATE TABLE hive_metastore.apa_prod.fat_roads
DEEP CLONE delta.`<prod-table-location-from-DESCRIBE-DETAIL>`;

---
## Option C — Delta Sharing (only if source is in Unity Catalog)

The dev workspace shows "Shares Received", so Delta Sharing is set up — but it only works if the
source table lives in a **Unity Catalog** catalog. `input_database_incremental` / `hive_metastore`
tables are **not** UC, so this won't apply unless you first register `fat_roads` in a UC catalog on
prod. Not worth it for a one-off copy; use **Option A / B**.

---
## Which to use

- **One-off copy, dev can reach prod's storage** → **Option B** (one command, no staging).
- **One-off / recurring, isolated environments** → **Option A** (staging path both sides own).